# Day 4 — Trees, Forests, SVMs & k-NN

### Data Prepararion:

In [39]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [40]:
Bank_data = pd.read_csv("Bank_Customer_Churn_Prediction.csv")
Bank_data

,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [41]:
Bank_data = pd.get_dummies(Bank_data,columns=['country'], drop_first=True)
Bank_data['gender'] = Bank_data['gender'].map({'Male': 0, 'Female': 1})

In [42]:
x = Bank_data.drop(["churn"], axis = 1)
y = Bank_data["churn"]

In [43]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Step 1 - Decision Tree, Random Forest, SVM & k-NN Training

### Decision Tree:

In [44]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

tree = DecisionTreeClassifier(max_depth=5, random_state=42)
tree.fit(X_train_scaled, y_train)

y_train_pred= tree.predict(X_train_scaled)
y_test_pred = tree.predict(X_test_scaled)

print(f"Train score: {accuracy_score(y_train_pred, y_train)}")
print(f"Test score: {accuracy_score(y_test_pred, y_test)}")

Train score: 0.858875
Test score: 0.8575


The test and train scores are very close which shows that the Tree experiences no overfitting, good in generalizing, and the depth is perfect.

### Random Forest:

In [45]:
from sklearn.ensemble import RandomForestClassifier

rfst = RandomForestClassifier(n_estimators=100, random_state=42)
rfst.fit(X_train, y_train)

y_train_pred= rfst.predict(X_train)
y_test_pred = rfst.predict(X_test)

print(f"Train score: {accuracy_score(y_train_pred, y_train)}")
print(f"Test score: {accuracy_score(y_test_pred, y_test)}")

print(rfst.feature_importances_)

Train score: 1.0
Test score: 0.8665
[0.14308275 0.0187304  0.23922656 0.08121328 0.14134001 0.13028337
 0.01922613 0.04171338 0.14596225 0.0257015  0.01352037]


The train score is perfect and the test score is the best of all other algorithms.
The gap between the scores is due to overfitting.

### SVM:

In [46]:
from sklearn.svm import SVC
supvecm = SVC(kernel="rbf")
supvecm.fit(X_train_scaled, y_train)

y_train_pred= supvecm.predict(X_train_scaled)
y_test_pred = supvecm.predict(X_test_scaled)

print(f"Train score: {accuracy_score(y_train_pred, y_train)}")
print(f"Test score: {accuracy_score(y_test_pred, y_test)}")


Train score: 0.865375
Test score: 0.856


The scores' gap is small indicating good generalization be the algorithm.

### K-NN:

In [47]:
from sklearn.neighbors import KNeighborsClassifier
Knn = KNeighborsClassifier(n_neighbors=5)
Knn.fit(X_train_scaled, y_train)

y_train_pred= Knn.predict(X_train_scaled)
y_test_pred = Knn.predict(X_test_scaled)

print(f"Train score: {accuracy_score(y_train_pred, y_train)}")
print(f"Test score: {accuracy_score(y_test_pred, y_test)}")

Train score: 0.874125
Test score: 0.83


Due to the large data size, the gap between the scores increased making is noisier and less reliable.

## Step 2 - F1 Score Comparison & Evaluation

In [48]:
from sklearn.metrics import f1_score

In [49]:
results = {
    "Decision Tree": f1_score(y_test,tree.predict(X_test_scaled)),
    "Random Forest": f1_score(y_test,rfst.predict(X_test)),
    "SVM": f1_score(y_test,supvecm.predict(X_test_scaled)),
    "k-NN": f1_score(y_test,Knn.predict(X_test_scaled))
}

comparison = pd.DataFrame(results.items(), columns=["Model","F1 Score"]).sort_values("F1 Score", ascending=False)
comparison

,Model,F1 Score
1,Random Forest,0.578199
0,Decision Tree,0.522613
2,SVM,0.510204
3,k-NN,0.462025


    The F1 results in comparison to the accuracy scores show a noticeable drop, this drop is due to imbalanced data.
    Random forset shows the best performance among the other algorithms despite room for improvement. It combines multiple decision trees and combines their predicitons. It is also less sensitive to noise and captures complex relationships.
    Decicion tree is the weekest among all algorithms and is struggling to record minority class.

## Step 3 - Random Forest's Top Feature Importances

In [50]:
importances = pd.Series(rfst.feature_importances_, index=X_train.columns).sort_values(ascending=False)
importances

age                 0.239227
estimated_salary    0.145962
credit_score        0.143083
balance             0.141340
products_number     0.130283
tenure              0.081213
active_member       0.041713
country_Germany     0.025702
credit_card         0.019226
gender              0.018730
country_Spain       0.013520
dtype: float64

    Top three features dominating were age, salary and credit score. Age is a good sign for the customer's behaviour even though it creates some bias. Estimated salary is an indicator of financial capability.credit score is a measure of financial reliability.
    